# 17. Letter Combinations of a Phone Number

[Problem](https://leetcode.com/problems/letter-combinations-of-a-phone-number/) · difficulty: medium

Two solutions that differ by the order of two `for` clauses in a comprehension. Both are 0 ms on
the judge. One does 341 recursive calls where the other does 5.


In [ ]:
import pathlib, sys

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'lc').is_dir())
PROBLEM = ROOT / 'problems' / '0017-letter-combinations-of-a-phone-number'
sys.path.insert(0, str(ROOT))

from lc.harness import load_solutions

solutions = load_solutions(PROBLEM)
by_name = {s.__name__: s for s in solutions}
[s.__name__ for s in solutions]


## Summary

| Approach | Time | Space | Recursive call sits in |
|---|---|---|---|
| `SolutionRecursiveSuffixPerLetter` | O(n²·4ⁿ) | O(4ⁿ) | the **inner** clause |
| `SolutionRecursiveSuffixOnce` | O(n·4ⁿ) | O(4ⁿ) | the **outer** clause |

Space is the answer itself; the recursion adds O(n) frames.


## The rule that makes them different

A nested comprehension re-evaluates its **inner** iterable once for every item of the outer one.
That is easy to forget when the inner iterable is a function call whose result does not depend on
the outer variable — it still gets called again.


In [ ]:
calls = []

def tail(label):
    calls.append(label)
    return ['x', 'y']

calls.clear()
_ = [f'{a}{b}' for a in 'abc' for b in tail('inner')]
print(f'recursive call in the INNER clause: evaluated {len(calls)} times')

calls.clear()
_ = [f'{a}{b}' for b in tail('outer') for a in 'abc']
print(f'recursive call in the OUTER clause: evaluated {len(calls)} times')


## What that costs in recursion

Counting actual calls into `letterCombinations`, by subclassing each solution and wrapping the
method — the classes in `solutions.py` are untouched.


In [ ]:
def count_calls(solution, digits):
    counted = {'n': 0}
    original = solution.letterCombinations

    def wrapper(self, d):
        counted['n'] += 1
        return original(self, d)

    instrumented = type(solution.__name__ + 'Counted', (solution,), {'letterCombinations': wrapper})
    instrumented().letterCombinations(digits)
    return counted['n']

lengths = [1, 2, 3, 4, 6, 8]
print(f"{'approach':<36}" + ''.join(f'{"n=" + str(k):>10}' for k in lengths))
for solution in solutions:
    row = ''.join(f'{count_calls(solution, "7" * k):10,}' for k in lengths)
    print(f'{solution.__name__:<36}{row}')


The branching version's call count is `(4ⁿ⁺¹ - 1) / 3` — it recomputes every suffix once per
letter, so the recursion tree has as many nodes as the answer has entries. The other makes
`n + 1` calls, walking straight down the string.

Note the constraint: `n ≤ 4`, so the judge only ever sees the fourth column. Both submissions
come back 0 ms.


In [ ]:
import time

def timed(solution, digits, runs=7):
    best = float('inf')
    for _ in range(runs):
        start = time.perf_counter()
        solution().letterCombinations(digits)
        best = min(best, (time.perf_counter() - start) * 1e3)
    return best

inputs = ['2345', '7979', '797979', '79797979']
print(f"{'approach':<36}" + ''.join(f'{d:>12}' for d in inputs))
for solution in solutions:
    row = ''.join(f'{timed(solution, d):9.3f} ms' for d in inputs)
    print(f'{solution.__name__:<36}{row}')
print()
print('answers identical:',
      sorted(solutions[0]().letterCombinations('7979')) == sorted(solutions[1]().letterCombinations('7979')))


## The other repeated work

Both rebuild the keypad dictionary on every call. How much that costs depends entirely on how
many calls there are — which is the thing the loop order decides.

Timed with `timeit` rather than a single run: at this scale a lone measurement varies by more
than the effect, and an earlier attempt at this comparison reversed the ranking.


In [ ]:
import timeit

KEYPAD = {'2': 'abc', '3': 'def', '4': 'ghi', '5': 'jkl',
          '6': 'mno', '7': 'pqrs', '8': 'tuv', '9': 'wxyz'}

class HoistedPerLetter:
    """The branching approach, keypad lifted to module scope."""

    def letterCombinations(self, digits):
        if digits == '':
            return ['']
        return [f'{letter}{suffix}'
                for letter in KEYPAD[digits[0]]
                for suffix in self.letterCombinations(digits[1:])]


class HoistedOnce:
    """The linear approach, keypad lifted to module scope."""

    def letterCombinations(self, digits):
        if digits == '':
            return ['']
        return [f'{letter}{suffix}'
                for suffix in self.letterCombinations(digits[1:])
                for letter in KEYPAD[digits[0]]]


def micros(obj, digits='7979'):
    runs = min(timeit.repeat(lambda: obj.letterCombinations(digits), number=1000, repeat=7))
    return runs / 1000 * 1e6

pairs = {
    'SuffixPerLetter (341 calls)': (by_name['SolutionRecursiveSuffixPerLetter'](), HoistedPerLetter()),
    'SuffixOnce (5 calls)': (by_name['SolutionRecursiveSuffixOnce'](), HoistedOnce()),
}
for label, (as_written, hoisted) in pairs.items():
    rebuilt, lifted = micros(as_written), micros(hoisted)
    print(f'{label:<30} rebuilt {rebuilt:7.2f} us   hoisted {lifted:7.2f} us   ({rebuilt / lifted:.2f}x)')


The same edit is worth 2.3x to the branching version and nothing to the linear one, because
the branching version builds that dictionary 341 times and the linear one builds it 5 times.

Per-call setup is only worth attacking once you know how often the call happens — fix the loop
order first and this optimization all but disappears.


## Takeaway

- In a comprehension, the inner iterable is re-evaluated once per outer item. Putting a recursive
  call there turns a linear walk into a branching tree — 341 calls against 5 at four digits.
- The judge could not see it. `n ≤ 4` caps the answer at 256 strings, so both submissions report
  0 ms; only a local measurement with inputs past the constraints separates them.
- Name loop variables for what they hold. The letters loop was called `digit`, which is exactly
  the kind of thing that hides which clause is which when there are two.
